In [1]:
import numpy as np 
import pandas as pd
from dataset import prepare_pinn_data,generate_dataset,generate_case
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

STIFFNESS VERIFICATION

Healthy stiffness:
[1000. 1000. 1000.]

Damaged stiffness:
[1000.  700. 1000.]

Healthy K:
[[ 2000. -1000.     0.]
 [-1000.  2000. -1000.]
 [    0. -1000.  1000.]]

Damaged K:
[[ 1700.  -700.     0.]
 [ -700.  1700. -1000.]
 [    0. -1000.  1000.]]

Maximum healthy residual: 1.776357e-15
Maximum damaged residual: 1.776357e-15


In [2]:
N = 10

masses = [1] * N
stiffness = [1000] * N
damping = [2] * N

sensor_dofs = [0, 2, 4, 5, 7, 9]

forces = [
    {
        "floor": 0,
        "amplitude": 10.0,
        "angular_frequency": 5.0
    },
    {
        "floor": 1,
        "amplitude": 8.0,
        "angular_frequency": 4.0
    },
    {
        "floor": 2,
        "amplitude": 12.0,
        "angular_frequency": 6.0
    },
    {
        "floor": 4,
        "amplitude": 7.0,
        "angular_frequency": 3.0
    },
    {
        "floor": 5,
        "amplitude": 9.0,
        "angular_frequency": 5.0
    },
    {
        "floor": 7,
        "amplitude": 6.0,
        "angular_frequency": 2.0
    },
    {
        "floor": 9,
        "amplitude": 11.0,
        "angular_frequency": 4.0
    }
]

In [3]:
healthy_case = generate_case(
    masses=masses,
    stiffness=stiffness,
    damping=damping,
    damage_element=0,
    damage_severity=0.0,
    forces=forces
)

healthy_case["damage_element"] = None
healthy_case["damage_severity"] = 0.0
healthy_case["is_healthy"] = True

In [4]:
t = healthy_case["time"]
x = healthy_case["displacement"]
v = healthy_case["velocity"]
a = healthy_case["acceleration"]

In [5]:
x_df = pd.DataFrame(x,columns=[f"Floor {i+1}" for i in range(N)])
x_df.insert(0,"time",t)

x_df.head(10)

,time,Floor 1,Floor 2,Floor 3,Floor 4,Floor 5,Floor 6,Floor 7,Floor 8,Floor 9,Floor 10
0,0.000000,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00
1,0.002500,1.292179e-07,8.425384e-08,1.858522e-07,1.415668e-09,5.475052e-08,1.161779e-07,8.699079e-10,3.088936e-08,8.527744e-10,1.139564e-07
2,0.005001,1.034474e-06,6.734311e-07,1.488019e-06,1.034281e-08,4.379728e-07,9.301608e-07,6.357778e-09,2.473635e-07,6.229724e-09,9.121248e-07
3,0.007501,3.484161e-06,2.278464e-06,5.010322e-06,4.450256e-08,1.478525e-06,3.132011e-06,2.726675e-08,8.326014e-07,2.679650e-08,3.073709e-06
4,0.010001,8.223843e-06,5.428800e-06,1.181912e-05,1.523086e-07,3.506334e-06,7.388673e-06,9.309779e-08,1.962229e-06,9.169367e-08,7.263051e-06
5,0.012502,1.596236e-05,1.068496e-05,2.292008e-05,4.318039e-07,6.852736e-06,1.433003e-05,2.639153e-07,3.798749e-06,2.599969e-07,1.412087e-05
6,0.015002,2.735778e-05,1.865315e-05,3.923374e-05,1.050396e-06,1.185054e-05,2.453452e-05,6.429600e-07,6.484683e-06,6.326389e-07,2.425570e-05
7,0.017502,4.310948e-05,2.989550e-05,6.175202e-05,2.117533e-06,1.883293e-05,3.862169e-05,1.295660e-06,1.018394e-05,1.275253e-06,3.830073e-05
8,0.020003,6.387023e-05,4.500223e-05,9.139422e-05,3.802334e-06,2.813679e-05,5.716446e-05,2.322239e-06,1.505131e-05,2.289369e-06,5.685721e-05
9,0.022503,9.023591e-05,6.460050e-05,1.289901e-04,6.347353e-06,4.010302e-05,8.067902e-05,3.866306e-06,2.122856e-05,3.820750e-06,8.048802e-05


In [6]:
v_df = pd.DataFrame(v,columns=[f"Floor {i+1}" for i in range(N)])
v_df.insert(0,"time",t)

v_df.head()

,time,Floor 1,Floor 2,Floor 3,Floor 4,Floor 5,Floor 6,Floor 7,Floor 8,Floor 9,Floor 10
0,0.000000,0.000000,0.000000,0.000000,0.000000e+00,0.000000,0.000000,0.000000e+00,0.000000,0.000000e+00,0.000000
1,0.002500,0.000156,0.000100,0.000224,6.436659e-07,0.000066,0.000140,3.952443e-07,0.000037,3.876692e-07,0.000137
2,0.005001,0.000621,0.000404,0.000893,6.291953e-06,0.000263,0.000558,3.855644e-06,0.000148,3.788657e-06,0.000547
3,0.007501,0.001388,0.000916,0.001995,2.494325e-05,0.000592,0.001247,1.525195e-05,0.000331,1.501678e-05,0.001226
4,0.010001,0.002450,0.001642,0.003518,6.786080e-05,0.001053,0.002200,4.142610e-05,0.000583,4.085038e-05,0.002168


In [7]:
a_df = pd.DataFrame(x,columns=[f"Floor {i+1}" for i in range(N)])
a_df.insert(0,"time",t)

a_df.head()

,time,Floor 1,Floor 2,Floor 3,Floor 4,Floor 5,Floor 6,Floor 7,Floor 8,Floor 9,Floor 10
0,0.000000,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00
1,0.002500,1.292179e-07,8.425384e-08,1.858522e-07,1.415668e-09,5.475052e-08,1.161779e-07,8.699079e-10,3.088936e-08,8.527744e-10,1.139564e-07
2,0.005001,1.034474e-06,6.734311e-07,1.488019e-06,1.034281e-08,4.379728e-07,9.301608e-07,6.357778e-09,2.473635e-07,6.229724e-09,9.121248e-07
3,0.007501,3.484161e-06,2.278464e-06,5.010322e-06,4.450256e-08,1.478525e-06,3.132011e-06,2.726675e-08,8.326014e-07,2.679650e-08,3.073709e-06
4,0.010001,8.223843e-06,5.428800e-06,1.181912e-05,1.523086e-07,3.506334e-06,7.388673e-06,9.309779e-08,1.962229e-06,9.169367e-08,7.263051e-06


In [8]:
t_tf = tf.convert_to_tensor(t.reshape(-1,1),dtype=tf.float32)
x_tf = tf.convert_to_tensor(x,dtype=tf.float32)
v_tf = tf.convert_to_tensor(v,dtype=tf.float32)
a_tf = tf.convert_to_tensor(a,dtype=tf.float32)


In [9]:
x_tf

<tf.Tensor: shape=(8000, 10), dtype=float32, numpy=
array([[ 0.      ,  0.      ,  0.      , ...,  0.      ,  0.      ,
         0.      ],
       [ 0.      ,  0.      ,  0.      , ...,  0.      ,  0.      ,
         0.      ],
       [ 0.000001,  0.000001,  0.000001, ...,  0.      ,  0.      ,
         0.000001],
       ...,
       [-0.010551, -0.016035, -0.013877, ..., -0.036048, -0.044641,
        -0.054542],
       [-0.011396, -0.017763, -0.016528, ..., -0.042384, -0.051377,
        -0.061443],
       [-0.012242, -0.019494, -0.019184, ..., -0.048725, -0.05812 ,
        -0.068347]], shape=(8000, 10), dtype=float32)>

In [10]:
print("t:", t_tf.shape)
print("x:", x_tf.shape)
print("v:", v_tf.shape)
print("a:", a_tf.shape)

t: (8000, 1)
x: (8000, 10)
v: (8000, 10)
a: (8000, 10)


In [11]:
type(x_tf)

tensorflow.python.framework.ops.EagerTensor

In [ ]:
class StructuralPINN(keras.Model):

    def __init__(self, dof):
        super().__init__()

        self.network = keras.Sequential([
            layers.Input(shape=(1,)),

            layers.Dense(64, activation="tanh"),
            layers.Dense(64, activation="tanh"),
            layers.Dense(64, activation="tanh"),
            layers.Dense(64, activation="tanh"),

            layers.Dense(dof)
        ])

    def call(self, t):
        return self.network(t)


def prepare_tensors(t,x):
    t_tf = tf.convert_to_tensor(t.reshape(-1,1),dtype=tf.float32)
    x_tf = tf.convert_to_tensor(x,dtype=tf.float32)

    return t_tf,x_tf

def calculate_response_derivatives(model, t):

    with tf.GradientTape(persistent=True) as tape2:
        tape2.watch(t)

        with tf.GradientTape() as tape1:
            tape1.watch(t)

            x_hat = model(t)

        # dx/dt for every floor
        v_hat = tape1.batch_jacobian(x_hat, t)[:, :, 0]

    # d²x/dt² for every floor
    a_hat = tape2.batch_jacobian(v_hat, t)[:, :, 0]

    del tape2

    return x_hat, v_hat, a_hat

def calculate_forces(forces_dict, N, t):

    t = tf.reshape(t, [-1])

    forces = tf.zeros(
        (tf.shape(t)[0], N),
        dtype=tf.float32
    )

    for item in forces_dict:

        floor = item["floor"]
        amplitude = item["amplitude"]
        angular_frequency = item["angular_frequency"]

        force = amplitude * tf.sin(
            angular_frequency * t
        )

        indices = tf.range(tf.shape(t)[0])

        scatter_indices = tf.stack(
            [
                indices,
                tf.fill([tf.shape(t)[0]], floor)
            ],
            axis=1
        )

        forces = tf.tensor_scatter_nd_add(
            forces,
            scatter_indices,
            force
        )

    return forces


def calculate_loss_function(
    model,
    t,
    x_true,
    M,
    C,
    K,
    forces,
    lr_data=1.0,
    lr_phy=1.0,
    lr_ic=1.0
):

    (
        x_hat,
        v_hat,
        a_hat,
        residual
    ) = calculate_physics_residual(
        model=model,
        t=t,
        M=M,
        C=C,
        K=K,
        forces=forces
    )

    # Displacement supervision
    data_loss = tf.reduce_mean(
        tf.square(x_hat - x_true)
    )

    # Physics equation:
    # Mx'' + Cx' + Kx - F = 0
    physics_loss = tf.reduce_mean(
        tf.square(residual)
    )

    # Initial conditions
    t0 = tf.zeros((1, 1), dtype=tf.float32)

    x0_hat, v0_hat, _ = calculate_response_derivatives(
        model,
        t0
    )

    ic_displacement_loss = tf.reduce_mean(
        tf.square(x0_hat)
    )

    ic_velocity_loss = tf.reduce_mean(
        tf.square(v0_hat)
    )

    ic_loss = (
        ic_displacement_loss +
        ic_velocity_loss
    )

    total_loss = (
        lr_data * data_loss +
        lr_phy * physics_loss +
        lr_ic * ic_loss
    )

    return (
        total_loss,
        data_loss,
        physics_loss,
        ic_loss
    )



@tf.function
def train_step(
    model,
    optimizer,
    t,
    x_true,
    M,
    C,
    K,
    forces,
    lr_data,
    lr_phy,
    lr_ic
):

    with tf.GradientTape() as tape:

        (
            total_loss,
            data_loss,
            physics_loss,
            ic_loss
        ) = calculate_loss_function(
            model=model,
            t=t,
            x_true=x_true,
            M=M,
            C=C,
            K=K,
            forces=forces,
            lr_data=lr_data,
            lr_phy=lr_phy,
            lr_ic=lr_ic
        )

    gradients = tape.gradient(
        total_loss,
        model.trainable_variables
    )

    optimizer.apply_gradients(
        zip(
            gradients,
            model.trainable_variables
        )
    )

    return (
        total_loss,
        data_loss,
        physics_loss,
        ic_loss
    )

def train_pinn(model,t,x_true,M,C,K,forces,epochs = 2000,learning_rate = 0.001,lr_data = 1.0,lr_phy = 1.0,lr_ic = 1.0):
    optimizer = keras.optimizers.Adam(learning_rate=learning_rate)

    for epoch in range(epochs):
        (total_loss,data_loss,physics_loss,ic_loss) = train_step(model= model,optimizer=optimizer,t = t,x_true=x_true,M=M,C=C,K=K,forces=forces,lr_data=lr_data,lr_phy=lr_phy,lr_ic=lr_ic)
        if epoch % 100 == 0:

            print(
                f"Epoch {epoch:4d} | "
                f"Total: {total_loss.numpy():.8e} | "
                f"Data: {data_loss.numpy():.8e} | "
                f"Physics: {physics_loss.numpy():.8e} | "
                f"IC: {ic_loss.numpy():.8e}"
            )
    

def predict_response(model, t):

    (
        x_hat,
        v_hat,
        a_hat
    ) = calculate_response_derivatives(
        model,
        t
    )

    return (
        x_hat.numpy(),
        v_hat.numpy(),
        a_hat.numpy()
    )  

In [13]:
x = tf.Variable(3.0)

with tf.GradientTape() as tape:
    y = x ** 2

gradient = tape.gradient(y, x)

print(y)
print(gradient)

tf.Tensor(9.0, shape=(), dtype=float32)
tf.Tensor(6.0, shape=(), dtype=float32)


In [14]:
df = pd.DataFrame(calculate_forces(forces,10,t_tf))
df.head()

,0,1,2,3,4,5,6,7,8,9
0,0.000000,0.000000,0.000000,0.0,0.000000,0.000000,0.0,0.000000,0.0,0.000000
1,0.125012,0.080009,0.180016,0.0,0.052506,0.112511,0.0,0.030004,0.0,0.110012
2,0.250005,0.160009,0.359991,0.0,0.105009,0.225005,0.0,0.060007,0.0,0.220013
3,0.374959,0.239994,0.539885,0.0,0.157506,0.337463,0.0,0.090008,0.0,0.329992
4,0.499854,0.319955,0.719658,0.0,0.209995,0.449869,0.0,0.120007,0.0,0.439938


In [15]:
print("TensorFlow version:", tf.__version__)
print("GPUs:", tf.config.list_physical_devices("GPU"))

TensorFlow version: 2.21.0
GPUs: []
